# GAN for SQLi — SeqGAN Master (Phase 2B) + SeqGAN Improved (Phase 3) — GPU

Notebook này **mở rộng trực tiếp** `GAN_SQLi_Colab_SeqGAN_only_phase1_2A_mini_GPU.ipynb` sang
Phase 2B và Phase 3, **chỉ chạy `seqgan_master` (Phase 2B) và `seqgan_improved` (Phase 3)**.
Không đụng SMOTE/GAN/CTGAN.

## Vì sao notebook này KHÔNG dùng `rank-phase2a` / `select-ratio` chính thức

Theo đúng thiết kế repo, `top2_scenarios_per_family.csv` phải ra từ Borda **chéo cả 4 method**,
và `select-ratio` phải xét viability **chéo cả 4 method** ở Phase 2B. Hiện tại:

- `gan` (vanilla GAN) đang collapse gần như tuyệt đối ở Phase 2A mini (dominant_payload_share
  tới 0.985) → không dùng được để rank.
- `seqgan_master` chạy mini (rollout 3, adv 6 epoch) sinh toàn garbage
  (`garbage_rate` 0.42–1.0, `family_motif_hit_rate` ≈ 0 ở hầu hết cell) → cũng không dùng được.
- Chỉ có `smote` và `ctgan` cho tín hiệu chất lượng tạm tin được.

**Quyết định (chủ động hạ tiêu chuẩn vì deadline gấp, đã thống nhất với người dùng):**

1. `top2_scenarios_per_family.csv` được viết **thủ công** dựa trên rank riêng của SMOTE + CTGAN
   (Borda 2 method thay vì 4). Xem bảng ở mục 5.
2. `selected_global_ratio.json` cho Phase 2B **được chọn thủ công** sau khi xem quality/RF của
   riêng `seqgan_master` quét qua 7 ratio, thay vì để `select-ratio` tự động (vì lệnh đó đòi cả
   4 method có mặt ở Phase 2B, ở đây ta chỉ chạy 1 method).
3. Hyperparameter SeqGAN được nâng lên một **tầng trung gian** — cao hơn hẳn bản mini (đã ra
   garbage) nhưng thấp hơn spec khóa trong README (120 pretrain / 200 adv / rollout 16, vì bản
   full từng treo 10h11' mới xong 2/200 epoch trên Colab GPU). Xem mục 6.

**Đây không phải kết quả nghiên cứu chính thức của repo.** Đây là một nhánh chạy nhanh, có chủ
đích hạ chuẩn, để có dataset/variant khả dụng đúng hạn. Khi có thời gian, nên chạy lại
`rank-phase2a` + `select-ratio` thật (đủ 4 method, đủ epoch) và đối chiếu.

Mặc định mọi cờ `RUN_*` là `False`.


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


## 2. Copy project và nhận diện runtime

In [ ]:
import csv
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import psutil
import torch
import yaml

DRIVE_MOUNT = Path("/content/drive/MyDrive")
DRIVE_PROJECT = DRIVE_MOUNT / "GAN_for_SQLi"
DRIVE_ROOT = DRIVE_MOUNT / "GAN_SQLi_Colab"
RESULTS_ROOT = DRIVE_ROOT / "results_seqgan_phase2b_phase3"
LOCAL_PROJECT = Path("/content/GAN_for_SQLi")

assert DRIVE_MOUNT.is_dir(), "Google Drive chưa mount"
assert (DRIVE_PROJECT / "scripts/research_pipeline.py").is_file(), (
    f"Không tìm thấy project tại {DRIVE_PROJECT}"
)

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(
    DRIVE_PROJECT,
    LOCAL_PROJECT,
    dirs_exist_ok=True,
    ignore=shutil.ignore_patterns(
        "__pycache__", "*.pyc", "results", "results_smoke", "results_mini",
        "results_cpu20_phase1_2a", "results_seqgan_only_mini",
        "results_seqgan_phase2b_phase3",
    ),
)
os.chdir(LOCAL_PROJECT)

print("Project:", LOCAL_PROJECT)
print("Results:", RESULTS_ROOT)
print("Python:", platform.python_version())
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), (
    "Notebook này yêu cầu runtime GPU (A100/L4/T4) vì seqgan_master/seqgan_improved chạy trên GPU"
)


## 3. Cài dependency

In [ ]:
%pip install -q -r requirements.txt


## 4. Cảnh báo thời lượng phiên GPU

Lần chạy full-spec trước (`rollout=16`, `adv=200`) mất **10h11'** mà mới xong 2/200 epoch
adversarial rồi bị đứt phiên — không có checkpoint nên không resume được. Notebook này:

- Dùng hyperparameter đã hạ tầng (mục 6), không phải full-spec, để một cell có cơ hội chạy xong
  trong 1 phiên Colab.
- Chạy `run-matrix --resume` cho từng shard riêng lẻ (giống notebook mini) để nếu một số shard
  xong, phần còn lại vẫn resume được ở phiên sau — nhưng **bản thân một shard SeqGAN vẫn không
  checkpoint theo epoch giữa chừng** (README: checkpoint theo epoch chỉ bật khi chạy standalone
  với `--checkpoint-dir`, orchestrator mặc định không bật). Nếu 1 shard bị đứt giữa chừng, shard
  đó phải chạy lại từ đầu ở phiên sau.


## 5. `top2_scenarios_per_family.csv` — chọn thủ công từ rank SMOTE + CTGAN

Bảng dưới là kết quả Borda 2-method (SQL / Novelty / Diversity xếp hạng ngang nhau, gộp trung
bình rank giữa SMOTE và CTGAN) tính từ `quality_metrics.json` của Phase 2A mini (R50):

| Family | Top 1 | Top 2 |
|---|---|---|
| boolean | E | D |
| error | D | B |
| time | A | D |
| union | D | F |

`D` (IQR random) lọt top 2 ở cả 4 family — thành phần ổn định nhất. `A` (baseline) chỉ thắng ở
`time`. Ghi thẳng ra `RESULTS_ROOT/phase2a/top2_scenarios_per_family.csv` đúng schema mà
`materialize_phase2b`/`freeze_phase3` đọc (`family`, `scenario`, `rank`; các cột khác chỉ để
truy vết, pipeline không đọc).


In [ ]:
MANUAL_TOP2 = [
    {"family": "boolean", "scenario": "E", "rank": 1, "source": "smote_ctgan_borda_R50_mini"},
    {"family": "boolean", "scenario": "D", "rank": 2, "source": "smote_ctgan_borda_R50_mini"},
    {"family": "error",   "scenario": "D", "rank": 1, "source": "smote_ctgan_borda_R50_mini"},
    {"family": "error",   "scenario": "B", "rank": 2, "source": "smote_ctgan_borda_R50_mini"},
    {"family": "time",    "scenario": "A", "rank": 1, "source": "smote_ctgan_borda_R50_mini"},
    {"family": "time",    "scenario": "D", "rank": 2, "source": "smote_ctgan_borda_R50_mini"},
    {"family": "union",   "scenario": "D", "rank": 1, "source": "smote_ctgan_borda_R50_mini"},
    {"family": "union",   "scenario": "F", "rank": 2, "source": "smote_ctgan_borda_R50_mini"},
]

MANUAL_SELECTION_PATH = RESULTS_ROOT / "phase2a" / "top2_scenarios_per_family.csv"
MANUAL_SELECTION_PATH.parent.mkdir(parents=True, exist_ok=True)
with MANUAL_SELECTION_PATH.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["family", "scenario", "rank", "source"])
    writer.writeheader()
    writer.writerows(MANUAL_TOP2)

print("Đã ghi selection thủ công:", MANUAL_SELECTION_PATH)
display(pd.read_csv(MANUAL_SELECTION_PATH))


## 6. Cấu hình runtime — tầng "hạ chuẩn có kiểm soát"

So với 3 mốc đã biết (spec khóa trong README / mini đã chạy ra garbage), đây là tầng giữa:

| Tham số | Spec README | Mini (đã ra garbage) | **Notebook này** |
|---|---:|---:|---:|
| `generator_pretrain_epochs` (seqgan_master) | 120 | 15 | **60** |
| `discriminator_pretrain_steps` | 50 | 8 | **30** |
| `adversarial_epochs` (seqgan_master) | 200 | 6 | **60** |
| `rollout_count` | 16 | 3 | **8** |
| `n_samples` | 2000 | 200 | **500** |
| `self_bleu_sample_size` | 200 | 20 | **100** |
| RF `n_estimators` | 300 | 30 | **100** |
| Phase 3 `generator_pretrain_epochs` (V1–V8) | 120 / 160 | — | **60 / 80** (giữ nguyên tỉ lệ, chỉ chia đôi) |

Các cờ khác (`sequence_length=20`, `d_pretrain_epochs=3`, `d_steps=5`, `d_epochs=3`,
`sql_reward`, `tokenizer_mode`, `discriminator_reward_weight=0.70`) **giữ nguyên spec khóa**,
không đổi — đây là các yếu tố nghiên cứu (biến độc lập của factorial V1–V8), không phải chỗ để
tiết kiệm thời gian.

`phase2b.methods` bị giới hạn còn `["seqgan_master"]` để `matrix --phase phase2b` chỉ sinh
8 cell × 7 ratio = 56 dòng (không kéo theo SMOTE/GAN/CTGAN).


In [ ]:
base_config = yaml.safe_load(
    (LOCAL_PROJECT / "configs/experiment_config.yaml").read_text(encoding="utf-8")
)
config = json.loads(json.dumps(base_config))

# Giữ đủ bốn method khi tạo matrix vì pipeline kiểm tra hard-count; lọc SeqGAN sau.
config["phase1"]["methods"] = ["smote", "gan", "ctgan", "seqgan_master"]
config["phase2a"]["methods"] = ["smote", "gan", "ctgan", "seqgan_master"]
config["phase2a"]["ratio"] = 50
config["phase2b"]["methods"] = ["smote", "gan", "ctgan", "seqgan_master"]

config["generation"]["n_samples"] = 500

config["generation"]["seqgan_master"].update({
    "sequence_length": 20,                # baseline khóa, không đổi
    "generator_pretrain_epochs": 60,      # 120 -> 60
    "discriminator_pretrain_steps": 30,   # 50 -> 30
    "discriminator_pretrain_epochs": 3,   # giữ spec
    "adversarial_epochs": 60,             # 200 -> 60
    "generator_steps": 1,
    "discriminator_steps": 5,             # giữ spec
    "discriminator_epochs": 3,            # giữ spec
    "rollout_count": 8,                   # 16 -> 8
})

config["generation"]["seqgan_improved"].update({
    "discriminator_pretrain_steps": 30,
    "discriminator_pretrain_epochs": 3,
    "adversarial_epochs": 60,
    "generator_steps": 1,
    "discriminator_steps": 5,
    "discriminator_epochs": 3,
    "rollout_count": 8,
})

# Phase 3: giữ nguyên factorial V1-V8 (sequence_length, sql_reward, tokenizer_mode không đổi),
# chỉ chia đôi generator_pretrain_epochs (120->60, 160->80) để khả thi trong 1 phiên GPU.
for variant in config["phase3"]["variants"]:
    variant["generator_pretrain_epochs"] = int(variant["generator_pretrain_epochs"] // 2)

config["quality"]["self_bleu_sample_size"] = 100
config["quality"]["calibration_margin"] = 0.05
config["detector"].update({
    "n_estimators": 100,
    "class_weight": "balanced_subsample",
    "severe_macro_f1_drop": 0.10,
    "severe_attack_recall_drop": 0.15,
})
config["outputs"]["results_root"] = str(RESULTS_ROOT)

RUNTIME_CONFIG_DIR = Path("/content/colab_configs")
RUNTIME_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG = RUNTIME_CONFIG_DIR / "experiment_seqgan_phase2b_phase3.yaml"
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

PIPELINE = [sys.executable, "scripts/research_pipeline.py", "--config", str(RUNTIME_CONFIG)]

print("Runtime config:", RUNTIME_CONFIG)
print(yaml.safe_dump({
    "phase1": config["phase1"],
    "phase2a": config["phase2a"],
    "phase2b": config["phase2b"],
    "generation": {
        "n_samples": config["generation"]["n_samples"],
        "seqgan_master": config["generation"]["seqgan_master"],
        "seqgan_improved": config["generation"]["seqgan_improved"],
    },
    "phase3": config["phase3"],
    "quality": config["quality"],
    "detector": config["detector"],
}, sort_keys=False))


## 7. Scheduler thích nghi theo CPU/RAM/GPU

Giống hệt notebook mini: tăng trần song song mỗi 45 giây, dừng phát shard mới khi CPU/RAM/GPU
util hoặc GPU memory chạm 92%. Shard đang chạy vẫn được phép hoàn tất.


In [ ]:
RAMP_INTERVAL_SECONDS = 45
RESOURCE_LIMIT_PERCENT = 92.0
POLL_SECONDS = 15
MAX_CONCURRENCY = 20


def gpu_snapshot():
    if not torch.cuda.is_available():
        return {"gpu_util_percent": 0.0, "gpu_memory_percent": 0.0,
                "gpu_memory_used_mb": 0.0, "gpu_memory_total_mb": 0.0}
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=False,
    )
    if result.returncode != 0 or not result.stdout.strip():
        return {"gpu_util_percent": 0.0, "gpu_memory_percent": 0.0,
                "gpu_memory_used_mb": 0.0, "gpu_memory_total_mb": 0.0}
    util, used, total = [float(v.strip()) for v in result.stdout.splitlines()[0].split(",")]
    return {"gpu_util_percent": util, "gpu_memory_percent": 100.0 * used / max(total, 1.0),
            "gpu_memory_used_mb": used, "gpu_memory_total_mb": total}


def resource_snapshot():
    gpu = gpu_snapshot()
    return {"cpu_percent": psutil.cpu_percent(interval=0.25),
            "ram_percent": psutil.virtual_memory().percent, **gpu}


def saturated(snapshot):
    watched = [snapshot["cpu_percent"], snapshot["ram_percent"],
               snapshot["gpu_util_percent"], snapshot["gpu_memory_percent"]]
    return max(watched) >= RESOURCE_LIMIT_PERCENT


def read_matrix(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def filter_method(frame, method):
    result = frame.loc[frame["method"].eq(method)].copy()
    result = result.sort_values(["family", "scenario", "ratio", "variant_id", "run_id"])
    return result.reset_index(drop=True)


def write_one_row(row, path):
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def run_adaptive_shards(frame, phase_label):
    if frame.empty:
        print(f"{phase_label}: không có shard nào để chạy")
        return {"completed": [], "failed": []}

    shard_root = Path("/content/parallel_shards/seqgan_phase2b_phase3") / phase_label
    shard_root.mkdir(parents=True, exist_ok=True)
    audit_dir = RESULTS_ROOT / "_scheduler_audit" / phase_label
    audit_dir.mkdir(parents=True, exist_ok=True)
    frame.to_csv(audit_dir / "selected_matrix.csv", index=False)

    pending = list(enumerate(frame.to_dict("records")))
    running = {}
    completed = []
    failed = []
    samples = []

    allowed_concurrency = 1
    next_ramp_at = time.monotonic() + RAMP_INTERVAL_SECONDS
    started_at = time.monotonic()

    print(f"{phase_label}: selected={len(frame)}, initial_concurrency=1, "
          f"max={MAX_CONCURRENCY}, ramp={RAMP_INTERVAL_SECONDS}s")

    try:
        while pending or running:
            now = time.monotonic()
            snapshot = resource_snapshot()

            if now >= next_ramp_at:
                if saturated(snapshot):
                    print("Giữ trần shard: tài nguyên đã chạm ngưỡng 92%.", flush=True)
                else:
                    allowed_concurrency = min(MAX_CONCURRENCY, allowed_concurrency + 1)
                    print(f"Tăng trần shard lên {allowed_concurrency}.", flush=True)
                next_ramp_at += RAMP_INTERVAL_SECONDS

            sample = {"elapsed_seconds": round(now - started_at, 1),
                      "allowed_concurrency": allowed_concurrency,
                      "running": len(running), "pending": len(pending), **snapshot}
            samples.append(sample)
            print(f"t={sample['elapsed_seconds']:.0f}s | CPU={snapshot['cpu_percent']:.1f}% | "
                  f"RAM={snapshot['ram_percent']:.1f}% | GPU={snapshot['gpu_util_percent']:.1f}% | "
                  f"GPUmem={snapshot['gpu_memory_percent']:.1f}% | limit={allowed_concurrency} | "
                  f"running={len(running)} | pending={len(pending)}", flush=True)

            while pending and len(running) < allowed_concurrency and not saturated(snapshot):
                index, row = pending.pop(0)
                shard_path = shard_root / f"shard_{index:03d}.csv"
                log_path = shard_root / f"shard_{index:03d}.stdout.log"
                write_one_row(row, shard_path)
                log_handle = log_path.open("w", encoding="utf-8")
                command = [*PIPELINE, "run-matrix", "--matrix", str(shard_path),
                           "--steps", "all", "--execute", "--resume"]
                process = subprocess.Popen(
                    command, cwd=LOCAL_PROJECT, stdout=log_handle,
                    stderr=subprocess.STDOUT, text=True, env=os.environ.copy(),
                    start_new_session=True,
                )
                running[index] = {"process": process, "log_handle": log_handle,
                                   "log_path": log_path, "run_id": row["run_id"]}
                print("launched", row["run_id"], flush=True)
                snapshot = resource_snapshot()

            for index, item in list(running.items()):
                return_code = item["process"].poll()
                if return_code is None:
                    continue
                item["log_handle"].close()
                del running[index]
                record = {"run_id": item["run_id"], "return_code": int(return_code),
                          "log": str(item["log_path"])}
                (completed if return_code == 0 else failed).append(record)
                print("finished", record, flush=True)

            pd.DataFrame(samples).to_csv(audit_dir / "resource_history.csv", index=False)
            (audit_dir / "scheduler_state.json").write_text(
                json.dumps({"phase": phase_label, "selected": len(frame), "completed": completed,
                            "failed": failed, "running": [i["run_id"] for i in running.values()],
                            "pending": [r["run_id"] for _, r in pending]},
                           ensure_ascii=False, indent=2), encoding="utf-8",
            )

            if pending or running:
                time.sleep(POLL_SECONDS)
    except KeyboardInterrupt:
        print("Đã nhận KeyboardInterrupt; kết thúc các shard con.", flush=True)
        for item in running.values():
            item["process"].terminate()
            item["log_handle"].close()
        raise

    for log_path in shard_root.glob("*.stdout.log"):
        shutil.copy2(log_path, audit_dir / log_path.name)

    print(f"Completed: {len(completed)} | Failed: {len(failed)}")
    if failed:
        raise RuntimeError(f"Có shard lỗi: {failed[:10]}")
    return {"completed": completed, "failed": failed}


def show_selected_status(frame, phase_label):
    rows = []
    for row in frame.to_dict("records"):
        manifest = Path(row["out_dir"]) / "run_manifest.json"
        if manifest.exists():
            item = json.loads(manifest.read_text(encoding="utf-8"))
            rows.append({"run_id": row["run_id"], "family": row["family"],
                         "scenario": row["scenario"], "ratio": row.get("ratio", ""),
                         "variant_id": row.get("variant_id", ""),
                         "status": item.get("status"), "failed_step": item.get("failed_step", "")})
        else:
            rows.append({"run_id": row["run_id"], "family": row["family"],
                         "scenario": row["scenario"], "ratio": row.get("ratio", ""),
                         "variant_id": row.get("variant_id", ""),
                         "status": "not_started", "failed_step": ""})
    status = pd.DataFrame(rows)
    print(f"Status {phase_label}: {len(status)} shard")
    display(status)
    return status


## 8. Prepare data

In [ ]:
subprocess.run([*PIPELINE, "prepare-data"], check=True)
print("prepare-data completed")


## 9. Phase 1 — SeqGAN Master (1 run, để tính `validity_thresholds.json`)

Cần bước này vì `rank-phase3` (mục 15) đòi `validity_thresholds.json` từ `calibrate-phase1`,
và ngưỡng này được tính riêng theo `generation_kind` (`direct` cho SeqGAN). Không chạy lại
SMOTE/GAN/CTGAN vì `config["phase1"]["methods"]` đã giới hạn còn `seqgan_master`.


In [ ]:
subprocess.run([*PIPELINE, "matrix", "--phase", "phase1"], check=True)
PHASE1_MATRIX_PATH = RESULTS_ROOT / "phase1" / "run_matrix.csv"
phase1_matrix = filter_method(read_matrix(PHASE1_MATRIX_PATH), "seqgan_master")
print("Phase 1 seqgan_master rows:", len(phase1_matrix))
display(phase1_matrix[["run_id", "method", "family", "scenario", "ratio", "data_status"]])


In [ ]:
RUN_PHASE_1 = False
if RUN_PHASE_1:
    run_adaptive_shards(phase1_matrix, "phase1")
else:
    print("RUN_PHASE_1=False; chưa chạy model")
show_selected_status(phase1_matrix, "phase1")


## 10. Calibrate threshold từ Phase 1

In [ ]:
RUN_CALIBRATE_PHASE1 = False
threshold_path = RESULTS_ROOT / "phase1" / "validity_thresholds.json"
if RUN_CALIBRATE_PHASE1:
    subprocess.run([*PIPELINE, "calibrate-phase1"], check=True)
    print("calibrate-phase1 completed ->", threshold_path)
else:
    print("RUN_CALIBRATE_PHASE1=False; chưa calibrate")
print("Đã tồn tại:", threshold_path.exists())


## 11. Phase 2B — `prepare-phase2b` với selection thủ công

`prepare-phase2b --selection` trỏ thẳng vào file đã ghi ở mục 5, bỏ qua yêu cầu phải có
`rank-phase2a` thật chạy trước.


In [ ]:
assert MANUAL_SELECTION_PATH.exists(), (
    f"Không tìm thấy selection: {MANUAL_SELECTION_PATH}"
)

prepare_result = subprocess.run(
    [
        *PIPELINE,
        "prepare-phase2b",
        "--selection",
        str(MANUAL_SELECTION_PATH),
    ],
    cwd=LOCAL_PROJECT,
    capture_output=True,
    text=True,
    check=False,
)

print(prepare_result.stdout)
if prepare_result.stderr:
    print(prepare_result.stderr)
if prepare_result.returncode != 0:
    raise RuntimeError(
        f"prepare-phase2b thất bại: {prepare_result.returncode}"
    )

PREFLIGHT_2B_PATH = (
    LOCAL_PROJECT / "data" / "prepared" / "phase2b" / "preflight.csv"
)
assert PREFLIGHT_2B_PATH.exists(), (
    f"Không tạo được preflight: {PREFLIGHT_2B_PATH}"
)

preflight_2b = pd.read_csv(PREFLIGHT_2B_PATH)
print("Preflight status:")
print(preflight_2b["status"].value_counts(dropna=False))
display(preflight_2b)


In [ ]:
# preflight.csv nằm trong work_dir (data/prepared/phase2b/preflight.csv), không phải RESULTS_ROOT
PREFLIGHT_2B_PATH = LOCAL_PROJECT / "data" / "prepared" / "phase2b" / "preflight.csv"
if PREFLIGHT_2B_PATH.exists():
    display(pd.read_csv(PREFLIGHT_2B_PATH))
else:
    print("Chưa thấy preflight.csv, kiểm tra lại đường dẫn work_dir trong config.")


## 12. Phase 2B — matrix + chạy `seqgan_master` (8 cell × 7 ratio = 56 run)

`config["phase2b"]["methods"] = ["seqgan_master"]` nên `matrix --phase phase2b` chỉ sinh dòng
cho `seqgan_master`, không cần lọc thêm.


In [ ]:
matrix_result = subprocess.run(
    [*PIPELINE, "matrix", "--phase", "phase2b"],
    cwd=LOCAL_PROJECT,
    capture_output=True,
    text=True,
    check=False,
)

print(matrix_result.stdout)
if matrix_result.stderr:
    print(matrix_result.stderr)
if matrix_result.returncode != 0:
    raise RuntimeError(
        f"Tạo Phase 2B matrix thất bại: {matrix_result.returncode}"
    )

PHASE2B_MATRIX_PATH = RESULTS_ROOT / "phase2b" / "run_matrix.csv"
phase2b_all = read_matrix(PHASE2B_MATRIX_PATH)
phase2b_seqgan = filter_method(phase2b_all, "seqgan_master")
phase2b_matrix = phase2b_seqgan.loc[
    phase2b_seqgan["data_status"].eq("ready")
].copy()

print("Full matrix rows:", len(phase2b_all))
print("SeqGAN rows:", len(phase2b_seqgan))
print("SeqGAN ready rows:", len(phase2b_matrix))
print("Data status:")
print(phase2b_seqgan["data_status"].value_counts(dropna=False))

display(
    phase2b_seqgan[
        ["run_id", "method", "family", "scenario", "ratio", "data_status"]
    ]
)

assert len(phase2b_all) == 224, (
    f"Phase 2B matrix phải có 224 dòng, hiện có {len(phase2b_all)}"
)
assert not phase2b_matrix.empty, (
    "Không có Phase 2B SeqGAN matrix ở trạng thái ready. "
    "Xem preflight và data_status phía trên."
)


In [ ]:
RUN_PHASE_2B = False

if RUN_PHASE_2B:
    if phase2b_matrix.empty:
        raise RuntimeError(
            "Không có Phase 2B SeqGAN matrix ở trạng thái ready."
        )
    run_adaptive_shards(phase2b_matrix, "phase2b")
else:
    print("RUN_PHASE_2B=False; chưa chạy model")

show_selected_status(phase2b_matrix, "phase2b")


## 13. Tổng hợp Phase 2B theo ratio — để **tự chọn** `selected_global_ratio`

Vì `select-ratio` chính thức cần đủ 4 method, bước này chỉ tổng hợp quality/RF của riêng
`seqgan_master` theo từng ratio, để bạn tự quyết định ratio khả dụng nhất (giữ nguyên
`normalized_holdout_overlap`, `garbage_rate`, `family_motif_hit_rate`, `unique_rate` làm căn cứ
chính, giống các nhóm chỉ số dùng trong `rank-phase2a`).


In [ ]:
import glob

rows = []
for f in glob.glob(str(RESULTS_ROOT / "phase2b" / "seqgan_master" / "*" / "*" / "R*" / "quality_metrics.json")):
    try:
        rows.append(json.load(open(f)))
    except Exception:
        continue

if rows:
    df = pd.DataFrame(rows)
    cols = ["family", "scenario", "ratio", "n_generated", "garbage_rate",
            "family_motif_hit_rate", "unique_rate", "model_collapse_rate", "sql_structure_rate"]
    summary = df[cols].sort_values(["ratio", "family", "scenario"])
    display(summary)
    print("\nTrung bình theo ratio:")
    display(df.groupby("ratio")[["garbage_rate", "family_motif_hit_rate", "unique_rate",
                                   "model_collapse_rate"]].mean().round(3))
else:
    print("Chưa có quality_metrics.json nào — chạy Phase 2B (mục 12) trước.")


## 14. Ghi `selected_global_ratio.json` thủ công

Đặt `MANUAL_SELECTED_RATIO` sau khi xem bảng ở mục 13 (mặc định `"50"` — nhất quán với ratio đã
dùng ở Phase 2A). Giá trị hợp lệ: `full`, `10`, `20`, `50`, `100`, `200`, `500`.


In [ ]:
MANUAL_SELECTED_RATIO = "50"  # <-- đổi tay sau khi xem mục 13

assert MANUAL_SELECTED_RATIO in {"full", "10", "20", "50", "100", "200", "500"}

SELECTED_RATIO_PATH = RESULTS_ROOT / "phase2b" / "selected_global_ratio.json"
SELECTED_RATIO_PATH.parent.mkdir(parents=True, exist_ok=True)
SELECTED_RATIO_PATH.write_text(
    json.dumps({
        "selected_global_ratio": MANUAL_SELECTED_RATIO,
        "selection_method": "manual_seqgan_master_only",
        "note": "Bypass select-ratio chinh thuc (can du 4 method); chi xet vien mot seqgan_master.",
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Đã ghi:", SELECTED_RATIO_PATH)
print(SELECTED_RATIO_PATH.read_text(encoding="utf-8"))


## 15. `freeze-phase3` — đóng băng 8 dataset ở ratio đã chọn

In [ ]:
subprocess.run(
    [*PIPELINE, "freeze-phase3",
     "--selection", str(MANUAL_SELECTION_PATH),
     "--ratio", str(SELECTED_RATIO_PATH)],
    check=True,
)
print("freeze-phase3 completed")
FROZEN_MANIFEST = LOCAL_PROJECT / "data" / "prepared" / "frozen" / "dataset_manifest.json"
if FROZEN_MANIFEST.exists():
    print(json.dumps(json.loads(FROZEN_MANIFEST.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


## 16. Phase 3 — matrix + chạy `seqgan_improved` (8 cell × 8 variant = 64 run)

`phase3_matrix` luôn dùng method `seqgan_improved` bất kể `phase2b.methods`, nên không cần lọc.


In [ ]:
subprocess.run([*PIPELINE, "matrix", "--phase", "phase3"], check=True)
PHASE3_MATRIX_PATH = RESULTS_ROOT / "phase3" / "run_matrix.csv"
phase3_matrix = read_matrix(PHASE3_MATRIX_PATH)
print("Phase 3 seqgan_improved rows:", len(phase3_matrix))
display(phase3_matrix[["run_id", "family", "scenario", "ratio", "variant_id", "data_status"]])


In [ ]:
RUN_PHASE_3 = False
if RUN_PHASE_3:
    run_adaptive_shards(phase3_matrix, "phase3")
else:
    print("RUN_PHASE_3=False; chưa chạy model")
show_selected_status(phase3_matrix, "phase3")


## 17. `rank-phase3` — xếp hạng 8 variant SeqGAN Improved

Cần `validity_thresholds.json` (mục 10) và `top2_scenarios_per_family.csv` (mục 5) đã có.


In [ ]:
RUN_RANK_PHASE_3 = False
if RUN_RANK_PHASE_3:
    subprocess.run([*PIPELINE, "rank-phase3"], check=True)
    ranking_path = RESULTS_ROOT / "phase3" / "variant_ranking.csv"
    selected_path = RESULTS_ROOT / "phase3" / "selected_seqgan_variant.json"
    display(pd.read_csv(ranking_path))
    print(selected_path.read_text(encoding="utf-8"))
else:
    print("RUN_RANK_PHASE_3=False; chưa rank")


## Giới hạn diễn giải & bước tiếp theo

- **`top2_scenarios_per_family.csv` là thủ công**, chỉ dựa trên SMOTE + CTGAN (2/4 method).
  Nếu sau này GAN được train đủ epoch và không còn collapse, ranking scenario có thể đổi —
  nên chạy lại `rank-phase2a` thật (đủ 4 method) trước khi công bố kết quả chính thức.
- **`selected_global_ratio.json` là thủ công**, chỉ dựa trên viability riêng của
  `seqgan_master`, không phải viability chéo 4 method như `select-ratio` yêu cầu.
- Hyperparameter SeqGAN (pretrain 60, adv 60, rollout 8) là **tầng trung gian tự chọn**, chưa có
  bằng chứng thực nghiệm là đủ để thoát vùng garbage như bản mini — cần xem kỹ
  `garbage_rate`/`family_motif_hit_rate` ở mục 13 trước khi tin tưởng bất kỳ ratio nào.
- Phase 3 dùng `generator_pretrain_epochs` đã chia đôi cho toàn bộ 8 variant (giữ nguyên tỉ lệ
  factorial) — không phải spec khóa gốc của README.
- Khi có thời gian/tài nguyên GPU dài hơn, nên chạy lại toàn bộ với hyperparameter spec đầy đủ
  (README khóa: pretrain 120, adv 200, rollout 16) và ranking 4-method thật, rồi đối chiếu với
  kết quả hạ chuẩn ở đây để xem có đổi quyết định hay không.
